# CalibrateQwen 03: on-policy distillation
We sample student trajectories and minimize reverse KL against the teacher on the student's occupied output distribution.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass

In [ ]:
from training.train_on_policy import OnPolicyConfig, train_on_policy

SFT_CHECKPOINT = None  # Paste the final tinker:// checkpoint from notebook 01 for the combined recipe
config = OnPolicyConfig(
    log_path='/content/calibrate_qwen_runs/on_policy',
    model_name='Qwen/Qwen3.5-4B',
    teacher_model='Qwen/Qwen3.5-9B',
    groups_per_batch=16,
    group_size=2,
    max_examples=4000,
    max_tokens=192,
    learning_rate=1e-4,
    kl_penalty_coef=1.0,
    load_checkpoint_path=SFT_CHECKPOINT,
    max_steps=None,  # Set 3 for a paid pipeline smoke test
    wandb_project='calibrate-qwen' if os.environ.get('WANDB_API_KEY') else None,
    wandb_name='on_policy',
)
config

In [ ]:
await train_on_policy(config)